# Load and Visualize Assistant Axis Vectors

This notebook loads pre-computed vectors from the Assistant Axis HuggingFace dataset and visualizes them in the PCA landscape.

**Key features:**
- Load model and pre-computed axis from HuggingFace
- Load 275 character role vectors
- Fit PCA on role vectors
- Visualize in 2D and 3D interactive plots
- Check alignment of default assistant with the axis direction

In [1]:
import sys
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from huggingface_hub import hf_hub_download, snapshot_download

# Add assistant-axis to path
assistant_axis_path = Path("../../../../assistant-axis").resolve()
if assistant_axis_path.exists():
    sys.path.insert(0, str(assistant_axis_path))
    print(f"Added {assistant_axis_path} to Python path")
else:
    print(f"Warning: assistant-axis repo not found at {assistant_axis_path}")

# Import assistant_axis utilities
try:
    from assistant_axis import load_axis, get_config
    from assistant_axis.internals import ProbingModel
    print("✓ Successfully imported assistant_axis")
except ImportError as e:
    print(f"⚠ Could not import assistant_axis: {e}")
    print("Will continue with custom implementations")

# Suppress HF progress bars for cleaner output
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

/Users/irakl/Desktop/Projects/LASR/persona-shattering-lasr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Added /Users/irakl/Desktop/Projects/LASR/assistant-axis to Python path
✓ Successfully imported assistant_axis


## Configuration

In [2]:
# Model configuration
MODEL_ID = "google/gemma-2-27b-it"
MODEL_NAME = "gemma-2-27b"  # For HuggingFace dataset paths
TARGET_LAYER = 22  # Middle layer, recommended for Gemma-2-27b

# HuggingFace dataset
REPO_ID = "lu-christina/assistant-axis-vectors"

print(f"Model: {MODEL_ID}")
print(f"Target layer: {TARGET_LAYER}")
print(f"Dataset: {REPO_ID}")

Model: google/gemma-2-27b-it
Target layer: 22
Dataset: lu-christina/assistant-axis-vectors


## Helper Classes

In [3]:
class MeanScaler:
    """Mean centering scaler (following assistant-axis implementation)."""
    def __init__(self):
        self.mean = None
    
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        return self
    
    def transform(self, X):
        return X - self.mean
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

## Load Model and Tokenizer

Load the model using ProbingModel from assistant_axis.

In [4]:
# Load model (optional - only needed if you want to generate or extract activations)
# For visualization only, we can skip this step and just load the vectors

LOAD_MODEL = False  # Set to True if you need the model

if LOAD_MODEL:
    print(f"Loading model: {MODEL_ID}...")
    try:
        pm = ProbingModel(MODEL_ID)
        model = pm.model
        tokenizer = pm.tokenizer
        print(f"✓ Model loaded successfully")
        print(f"  Device: {next(model.parameters()).device}")
        print(f"  Dtype: {next(model.parameters()).dtype}")
    except Exception as e:
        print(f"⚠ Error loading model: {e}")
        model = None
        tokenizer = None
else:
    print("Skipping model loading (LOAD_MODEL=False)")
    print("For visualization only, we just need the pre-computed vectors")
    model = None
    tokenizer = None

Skipping model loading (LOAD_MODEL=False)
For visualization only, we just need the pre-computed vectors


## Download and Load Pre-computed Vectors

Download from HuggingFace:
- 275 character role vectors
- Default assistant vector
- Assistant axis

In [5]:
# Download assistant axis
print(f"Downloading assistant axis from {REPO_ID}...")
axis_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=f"{MODEL_NAME}/assistant_axis.pt",
    repo_type="dataset"
)
axis = load_axis(axis_path)
print(f"✓ Loaded assistant axis: {axis.shape}")
print(f"  Path: {axis_path}")

✓ Loaded assistant axis: torch.Size([46, 4608])
  Path: /Users/irakl/.cache/huggingface/hub/datasets--lu-christina--assistant-axis-vectors/snapshots/3b3b788432ad33e3a28d9ff08e88a530c0740814/gemma-2-27b/assistant_axis.pt


In [6]:
# Download all vectors (role vectors + default vector + axis)
print(f"\nDownloading all vectors from {REPO_ID}...")
local_dir = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns=[
        f"{MODEL_NAME}/role_vectors/*.pt",
        f"{MODEL_NAME}/default_vector.pt",
        f"{MODEL_NAME}/assistant_axis.pt",
    ]
)
print(f"✓ Downloaded to: {local_dir}")

✓ Downloaded to: /Users/irakl/.cache/huggingface/hub/datasets--lu-christina--assistant-axis-vectors/snapshots/3b3b788432ad33e3a28d9ff08e88a530c0740814


In [7]:
# Load role vectors
role_vectors_dir = Path(local_dir) / MODEL_NAME / "role_vectors"
role_vectors = {}
for pt_file in sorted(role_vectors_dir.glob("*.pt")):
    role_name = pt_file.stem
    role_vectors[role_name] = torch.load(pt_file, map_location="cpu", weights_only=False)

print(f"✓ Loaded {len(role_vectors)} role vectors")
print(f"  Example roles: {list(role_vectors.keys())[:10]}")
print(f"  Vector shape: {next(iter(role_vectors.values())).shape}")

✓ Loaded 275 role vectors
  Example roles: ['aberration', 'absurdist', 'accountant', 'activist', 'actor', 'addict', 'adolescent', 'advocate', 'alien', 'altruist']
  Vector shape: torch.Size([46, 4608])


In [8]:
# Load default vector
default_vector_path = Path(local_dir) / MODEL_NAME / "default_vector.pt"
default_vector = torch.load(default_vector_path, map_location="cpu", weights_only=False)
print(f"✓ Loaded default vector: {default_vector.shape}")

✓ Loaded default vector: torch.Size([46, 4608])


In [9]:
# Load assistant axis (already loaded above, but included for completeness)
assistant_axis_path = Path(local_dir) / MODEL_NAME / "assistant_axis.pt"
assistant_axis = torch.load(assistant_axis_path, map_location="cpu", weights_only=False)
print(f"✓ Loaded assistant axis: {assistant_axis.shape}")

✓ Loaded assistant axis: torch.Size([46, 4608])


## Fit PCA on Role Vectors

Extract vectors at target layer and fit PCA.

In [10]:
# Extract vectors at target layer
role_labels = list(role_vectors.keys())
role_vectors_array = torch.stack(
    [role_vectors[label][TARGET_LAYER] for label in role_labels]
).float().numpy()

print(f"Role vectors at layer {TARGET_LAYER}:")
print(f"  Shape: {role_vectors_array.shape}")
print(f"  Number of roles: {len(role_labels)}")

Role vectors at layer 22:
  Shape: (275, 4608)
  Number of roles: 275


In [11]:
# Fit scaler and PCA
print(f"\nFitting PCA on {len(role_labels)} vectors at layer {TARGET_LAYER}...")

scaler = MeanScaler()
role_vectors_scaled = scaler.fit_transform(role_vectors_array)

pca = PCA()
pca_result = pca.fit_transform(role_vectors_scaled)

variance_explained = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_explained)

print(f"✓ PCA fitted with {len(variance_explained)} components")
print(f"\nVariance explained by first 5 PCs:")
for i in range(min(5, len(variance_explained))):
    print(f"  PC{i+1}: {variance_explained[i]:.4f} ({variance_explained[i]*100:.2f}%)")
print(f"\nCumulative variance (first 3 PCs): {cumulative_variance[2]:.4f} ({cumulative_variance[2]*100:.2f}%)")

# DIAGNOSTIC: Check relationship in ORIGINAL SPACE (before PCA)
print("\n" + "="*60)
print("DIAGNOSTIC: Vector Relationship in ORIGINAL SPACE")
print("="*60)

# Get vectors in original space
default_orig = default_vector[TARGET_LAYER].float().numpy()
axis_orig = assistant_axis[TARGET_LAYER].float().numpy()
mean_roles_orig = role_vectors_array.mean(axis=0)

# Check: default ≈ mean(roles) + axis
computed_default_orig = mean_roles_orig + axis_orig
diff_orig = default_orig - computed_default_orig
relative_error = np.linalg.norm(diff_orig) / np.linalg.norm(default_orig) * 100

print(f"\nIn ORIGINAL (pre-PCA) space:")
print(f"  Default vector norm: {np.linalg.norm(default_orig):.2f}")
print(f"  Axis norm: {np.linalg.norm(axis_orig):.2f}")
print(f"  Mean roles norm: {np.linalg.norm(mean_roles_orig):.2f}")
print(f"\n  Computed (mean + axis) norm: {np.linalg.norm(computed_default_orig):.2f}")
print(f"  Difference: {np.linalg.norm(diff_orig):.6f}")
print(f"  Relative error: {relative_error:.6f}%")

if relative_error < 1.0:  # Less than 1% error
    print("\n✓ EXCELLENT MATCH in original space!")
    print("  The formula holds: default ≈ mean(roles) + axis")
    print("  (Small numerical error is expected due to floating point precision)")
else:
    print("\n⚠ Significant mismatch even in original space")
    print("  This suggests the vectors were computed with different data")

# DIAGNOSTIC: Check PC1 alignment with Assistant Axis
print("\n" + "="*60)
print("DIAGNOSTIC: PC1 vs Assistant Axis Alignment")
print("="*60)

# PC1 direction in original feature space
pc1_direction = pca.components_[0]

# Assistant axis in original feature space (after scaling)
assistant_axis_scaled = scaler.transform(assistant_axis[TARGET_LAYER].float().numpy().reshape(1, -1))[0]

# Compute cosine similarity in original space
cosine_sim_original = np.dot(pc1_direction, assistant_axis_scaled) / (
    np.linalg.norm(pc1_direction) * np.linalg.norm(assistant_axis_scaled)
)

print(f"\nCosine similarity (PC1 vs Assistant Axis in original space): {cosine_sim_original:.4f}")
print(f"Paper claims this should be > 0.71 for high alignment")
print(f"(Note: Negative value means opposite direction, check absolute value)")

if abs(cosine_sim_original) > 0.71:
    print(f"✓ HIGH alignment - PC1 captures the assistant axis direction")
    if cosine_sim_original < 0:
        print("  (PC1 points in opposite direction to axis)")
else:
    print("⚠ LOW alignment - PC1 does NOT align well with assistant axis")
    print("  This suggests:")
    print("  - The assistant axis is NOT the dominant source of variance")
    print("  - Other factors (e.g., different role characteristics) dominate PC1")
    print("  - This is a valid finding for this model/layer combination")


Fitting PCA on 275 vectors at layer 22...
✓ PCA fitted with 275 components

Variance explained by first 5 PCs:
  PC1: 0.4880 (48.80%)
  PC2: 0.0980 (9.80%)
  PC3: 0.0698 (6.98%)
  PC4: 0.0551 (5.51%)
  PC5: 0.0333 (3.33%)

Cumulative variance (first 3 PCs): 0.6558 (65.58%)

DIAGNOSTIC: Vector Relationship in ORIGINAL SPACE

In ORIGINAL (pre-PCA) space:
  Default vector norm: 16326.44
  Axis norm: 780.69
  Mean roles norm: 15860.69

  Computed (mean + axis) norm: 16306.54
  Difference: 21.060341
  Relative error: 0.128995%

✓ EXCELLENT MATCH in original space!
  The formula holds: default ≈ mean(roles) + axis
  (Small numerical error is expected due to floating point precision)

DIAGNOSTIC: PC1 vs Assistant Axis Alignment

Cosine similarity (PC1 vs Assistant Axis in original space): -0.6599
Paper claims this should be > 0.71 for high alignment
(Note: Negative value means opposite direction, check absolute value)
⚠ LOW alignment - PC1 does NOT align well with assistant axis
  This sugge

## Project Special Vectors

Project default assistant and assistant axis into PC space.

In [12]:
def project_vector(
    vector: torch.Tensor,
    scaler: MeanScaler,
    pca: PCA,
    is_difference_vector: bool = False
) -> np.ndarray:
    """Project a vector into PC space.

    Args:
        vector: The vector to project
        scaler: The MeanScaler fitted on role vectors
        pca: The PCA object fitted on centered role vectors
        is_difference_vector: If True, don't center the vector (it's already a difference)
    """
    if isinstance(vector, torch.Tensor):
        vector = vector.float().numpy()

    if is_difference_vector:
        # Don't center - the vector is already in the mean-centered coordinate system
        vector_pc = pca.transform(vector.reshape(1, -1))[0]
    else:
        # Center by subtracting the mean of roles
        vector_scaled = scaler.transform(vector.reshape(1, -1))
        vector_pc = pca.transform(vector_scaled)[0]

    return vector_pc

In [13]:
# Project special vectors
# NOTE: The assistant axis is a difference vector (default - mean(roles)),
# so we should NOT center it again when projecting to PC space
default_pc = project_vector(default_vector[TARGET_LAYER], scaler, pca, is_difference_vector=False)
assistant_axis_pc = project_vector(assistant_axis[TARGET_LAYER], scaler, pca, is_difference_vector=True)

print(f"Default assistant PC coordinates (first 5): {default_pc[:5]}")
print(f"Assistant axis PC coordinates (first 5): {assistant_axis_pc[:5]}")

# Compute distance from origin
default_dist = np.linalg.norm(default_pc[:3])
axis_dist = np.linalg.norm(assistant_axis_pc[:3])
print(f"\nDistance from origin (PC1-3):")
print(f"  Default assistant: {default_dist:.2f}")
print(f"  Assistant axis: {axis_dist:.2f}")

# DIAGNOSTIC: Check the mathematical relationship
print("\n" + "="*60)
print("DIAGNOSTIC: Vector Relationship (mean + axis = default?)")
print("="*60)

# Compute mean of roles in PC space
mean_roles_pc = np.mean(pca_result, axis=0)

# The expected relationship: default_pc ≈ mean_roles_pc + assistant_axis_pc
# Because axis = mean(assistant) - mean(roles)
# Therefore: mean(assistant) = mean(roles) + axis
computed_default = mean_roles_pc + assistant_axis_pc

print(f"\nActual Default PC (first 3):        {default_pc[:3]}")
print(f"Computed (mean + axis) (first 3):   {computed_default[:3]}")
print(f"Difference:                          {default_pc[:3] - computed_default[:3]}")
print(f"Difference magnitude:                {np.linalg.norm(default_pc[:3] - computed_default[:3]):.4f}")

# Check if they're close
if np.linalg.norm(default_pc[:3] - computed_default[:3]) < 1.0:
    print("\n✓ MATCH! default ≈ mean(roles) + axis")
    print("  This confirms: axis = mean(assistant) - mean(roles)")
else:
    print("\n⚠ MISMATCH! default ≠ mean(roles) + axis")
    print("  This suggests the axis or default were computed differently")
    print("  Possible reasons:")
    print("  - default_vector is not the mean of assistant activations")
    print("  - axis was computed with different assistant samples")
    print("  - Different preprocessing/centering was applied")

# DIAGNOSTIC: Check Default Assistant's position relative to PC1
print("\n" + "="*60)
print("DIAGNOSTIC: Default Assistant Position")
print("="*60)

# Check if default is at the PC1 extreme
pc1_values = pca_result[:, 0]  # All role vectors' PC1 coordinates
max_pc1 = np.max(pc1_values)
min_pc1 = np.min(pc1_values)
default_pc1 = default_pc[0]

print(f"\nPC1 range for role vectors: [{min_pc1:.2f}, {max_pc1:.2f}]")
print(f"Default assistant PC1: {default_pc1:.2f}")
print(f"\nDefault's position relative to PC1 range:")
print(f"  Distance from max: {max_pc1 - default_pc1:.2f}")
print(f"  Distance from min: {default_pc1 - min_pc1:.2f}")

if default_pc1 > max_pc1:
    print(f"✓ Default is BEYOND the maximum role PC1 (as expected)")
    print(f"  Exceeds max by: {default_pc1 - max_pc1:.2f}")
elif default_pc1 > max_pc1 - 0.03 * (max_pc1 - min_pc1):
    print(f"✓ Default is VERY CLOSE to maximum role PC1 (within 3% of range)")
else:
    print(f"⚠ Default is NOT at the PC1 extreme")
    print(f"  This suggests PC1 may not align with the assistant axis")

# Check alignment in PC space
print("\n" + "="*60)
print("DIAGNOSTIC: Assistant Axis Direction in PC Space")
print("="*60)

# In PC space, if assistant axis ≈ PC1, then assistant_axis_pc should be mostly [large, 0, 0, ...]
print(f"\nAssistant Axis PC coordinates (first 3): {assistant_axis_pc[:3]}")
print(f"\nIf aligned with PC1, we expect:")
print(f"  - Large PC1 component (currently: {assistant_axis_pc[0]:.2f})")
print(f"  - Small PC2 component (currently: {assistant_axis_pc[1]:.2f})")
print(f"  - Small PC3 component (currently: {assistant_axis_pc[2]:.2f})")

# Check the ratio
pc1_magnitude = abs(assistant_axis_pc[0])
other_magnitude = np.linalg.norm(assistant_axis_pc[1:3])
ratio = pc1_magnitude / (other_magnitude + 1e-10)

print(f"\nPC1 magnitude / Other PCs magnitude: {ratio:.2f}")
if ratio > 3.0:
    print("✓ Assistant axis is STRONGLY aligned with PC1")
elif ratio > 1.5:
    print("~ Assistant axis is MODERATELY aligned with PC1")
else:
    print("⚠ Assistant axis is NOT well aligned with PC1")
    print("  The red line in the plot should be diagonal, not horizontal")

Default assistant PC coordinates (first 5): [674.28394  -21.292511  65.77762   53.088142 -35.81048 ]
Assistant axis PC coordinates (first 5): [659.74725  -25.354748  65.80742   55.075027 -37.894268]

Distance from origin (PC1-3):
  Default assistant: 677.82
  Assistant axis: 663.51

DIAGNOSTIC: Vector Relationship (mean + axis = default?)

Actual Default PC (first 3):        [674.28394  -21.292511  65.77762 ]
Computed (mean + axis) (first 3):   [659.74725  -25.354746  65.8074  ]
Difference:                          [14.536682    4.062235   -0.02978516]
Difference magnitude:                15.0936

⚠ MISMATCH! default ≠ mean(roles) + axis
  This suggests the axis or default were computed differently
  Possible reasons:
  - default_vector is not the mean of assistant activations
  - axis was computed with different assistant samples
  - Different preprocessing/centering was applied

DIAGNOSTIC: Default Assistant Position

PC1 range for role vectors: [-1661.14, 1118.58]
Default assistant 

## Check Alignment: Does Default Lie on the Axis Line?

Key question: Is the default assistant vector aligned with the assistant axis direction?

In [14]:
# Compute alignment metrics
print("=" * 60)
print("ALIGNMENT ANALYSIS: Default vs Axis")
print("=" * 60)

# Normalize both vectors for comparison
axis_direction = assistant_axis_pc[:3] / np.linalg.norm(assistant_axis_pc[:3])
default_direction = default_pc[:3] / np.linalg.norm(default_pc[:3])

# Cosine similarity (alignment)
cosine_sim = np.dot(axis_direction, default_direction)
angle_deg = np.arccos(np.clip(cosine_sim, -1, 1)) * 180 / np.pi

print(f"\nCosine similarity: {cosine_sim:.4f}")
print(f"Angle between vectors: {angle_deg:.2f}°")
print(f"\nInterpretation:")
if abs(cosine_sim) > 0.99:
    print("  ✓ VERY STRONG alignment - vectors are nearly parallel")
elif abs(cosine_sim) > 0.9:
    print("  ✓ STRONG alignment - vectors point in similar directions")
elif abs(cosine_sim) > 0.7:
    print("  ~ MODERATE alignment")
else:
    print("  ✗ WEAK alignment - vectors are not well aligned")

if cosine_sim > 0:
    print("  → Same direction (default points toward positive axis)")
else:
    print("  → Opposite direction (default points toward negative axis)")

# Distance from axis line (perpendicular distance)
# Project default onto axis, then compute perpendicular distance
projection_length = np.dot(default_pc[:3], axis_direction)
projection_point = projection_length * axis_direction
perpendicular_dist = np.linalg.norm(default_pc[:3] - projection_point)

print(f"\nDistance from axis line: {perpendicular_dist:.2f}")
print(f"Relative to default magnitude: {perpendicular_dist / default_dist * 100:.1f}%")

# Where does default project onto the axis?
print(f"\nProjection onto axis: {projection_length:.2f}")
print(f"  (Axis extends from origin to {axis_dist:.2f})")
print(f"  Default projects to {projection_length / axis_dist * 100:.1f}% along the axis")

ALIGNMENT ANALYSIS: Default vs Axis

Cosine similarity: 1.0000
Angle between vectors: 0.41°

Interpretation:
  ✓ VERY STRONG alignment - vectors are nearly parallel
  → Same direction (default points toward positive axis)

Distance from axis line: 4.84
Relative to default magnitude: 0.7%

Projection onto axis: 677.80
  (Axis extends from origin to 663.51)
  Default projects to 102.2% along the axis


## Visualization Functions

In [15]:
def plot_2d_with_axis_line(
    pca_result: np.ndarray,
    role_labels: List[str],
    variance_explained: np.ndarray,
    default_pc: np.ndarray,
    assistant_axis_pc: np.ndarray,
    role_vectors_centered: np.ndarray,
    assistant_axis_orig: np.ndarray,
    test_vectors: Optional[Dict[str, np.ndarray]] = None,
    title: str = "Assistant Axis as Extended Line"
) -> go.Figure:
    """Plot with axis as a bidirectional line extending through the space.
    
    Args:
        pca_result: PCA-transformed role vectors
        role_labels: List of role names
        variance_explained: Variance explained by each PC
        default_pc: Default vector in PC space
        assistant_axis_pc: Assistant axis in PC space
        role_vectors_centered: Mean-centered role vectors in original space
        assistant_axis_orig: Assistant axis in original space
        test_vectors: Optional dict of test vectors to plot
        title: Plot title
    """
    fig = go.Figure()

    # Compute projections onto assistant axis (in original space)
    projections = role_vectors_centered @ assistant_axis_orig / np.linalg.norm(assistant_axis_orig)

    # Plot role vectors, colored by projection
    fig.add_trace(go.Scatter(
        x=pca_result[:, 0],
        y=pca_result[:, 1],
        mode='markers',
        marker=dict(
            size=6,
            color=projections,
            colorscale='RdBu_r',  # Red (positive/assistant-like) to Blue (negative/role-like)
            cmin=projections.min(),
            cmax=projections.max(),
            colorbar=dict(
                title="Projection onto<br>Assistant Axis",
                orientation='h',
                x=0.5,
                xanchor='center',
                y=-0.15,
                yanchor='top',
                len=0.6,
                thickness=15
            ),
            opacity=0.7
        ),
        text=role_labels,
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>Projection: %{marker.color:.2f}<extra></extra>',
        name='Role vectors'
    ))

    # Plot default assistant
    fig.add_trace(go.Scatter(
        x=[default_pc[0]],
        y=[default_pc[1]],
        mode='markers',
        marker=dict(size=20, color='gold', symbol='star', line=dict(color='black', width=2)),
        text=['Default Assistant'],
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>',
        name='Default Assistant'
    ))

    # Plot assistant axis as line through origin
    # Normalize and extend to cover the plot area
    axis_direction = assistant_axis_pc[:2] / np.linalg.norm(assistant_axis_pc[:2])
    
    # Determine how far to extend the line
    max_extent = max(np.abs(pca_result[:, :2]).max(), np.abs(default_pc[:2]).max()) * 1.5
    
    # Extend in both directions from origin
    line_start = -max_extent * axis_direction
    line_end = max_extent * axis_direction
    
    fig.add_trace(go.Scatter(
        x=[line_start[0], line_end[0]],
        y=[line_start[1], line_end[1]],
        mode='lines',
        line=dict(color='red', width=3, dash='solid'),
        name='Assistant Axis',
        hovertemplate='Assistant Axis Line<extra></extra>'
    ))
    
    # Add arrow at the positive end to show direction
    arrow_start = 0.8 * max_extent * axis_direction
    arrow_end = max_extent * axis_direction
    
    fig.add_trace(go.Scatter(
        x=[arrow_start[0], arrow_end[0]],
        y=[arrow_start[1], arrow_end[1]],
        mode='lines+markers',
        line=dict(color='red', width=3),
        marker=dict(size=[0, 15], color='red', symbol='arrow-up'),
        name='Axis Direction',
        hovertemplate='Toward Default<extra></extra>',
        showlegend=False
    ))
    
    # Add origin marker
    fig.add_trace(go.Scatter(
        x=[0],
        y=[0],
        mode='markers',
        marker=dict(size=10, color='black', symbol='circle'),
        text=['Origin'],
        hovertemplate='<b>Origin (mean of roles)</b><extra></extra>',
        name='Origin'
    ))
    
    # Plot test vectors if provided
    if test_vectors:
        colors = ['orange', 'cyan', 'magenta', 'brown', 'pink']
        for i, (name, pc_coords) in enumerate(test_vectors.items()):
            color = colors[i % len(colors)]
            fig.add_trace(go.Scatter(
                x=[pc_coords[0]],
                y=[pc_coords[1]],
                mode='markers',
                marker=dict(size=15, color=color, symbol='x', line=dict(color='black', width=2)),
                text=[name],
                hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>',
                name=name
            ))
    
    fig.update_layout(
        title=title,
        xaxis_title=f'PC1 ({variance_explained[0]:.1%} variance)',
        yaxis_title=f'PC2 ({variance_explained[1]:.1%} variance)',
        hovermode='closest',
        width=900,
        height=750,
        template='plotly_white',
        margin=dict(b=100)  # Add bottom margin for colorbar
    )
    
    return fig


def plot_3d_with_axis_line(
    pca_result: np.ndarray,
    role_labels: List[str],
    variance_explained: np.ndarray,
    default_pc: np.ndarray,
    assistant_axis_pc: np.ndarray,
    role_vectors_centered: np.ndarray,
    assistant_axis_orig: np.ndarray,
    test_vectors: Optional[Dict[str, np.ndarray]] = None,
    title: str = "Assistant Axis as Extended Line (3D)"
) -> go.Figure:
    """Plot 3D with axis as a bidirectional line.
    
    Args:
        pca_result: PCA-transformed role vectors
        role_labels: List of role names
        variance_explained: Variance explained by each PC
        default_pc: Default vector in PC space
        assistant_axis_pc: Assistant axis in PC space
        role_vectors_centered: Mean-centered role vectors in original space
        assistant_axis_orig: Assistant axis in original space
        test_vectors: Optional dict of test vectors to plot
        title: Plot title
    """
    fig = go.Figure()
    
    # Compute projections onto assistant axis (in original space)
    projections = role_vectors_centered @ assistant_axis_orig / np.linalg.norm(assistant_axis_orig)
    
    # Plot role vectors, colored by projection
    fig.add_trace(go.Scatter3d(
        x=pca_result[:, 0],
        y=pca_result[:, 1],
        z=pca_result[:, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=projections,
            colorscale='RdBu_r',  # Red (positive/assistant-like) to Blue (negative/role-like)
            cmin=projections.min(),
            cmax=projections.max(),
            colorbar=dict(
                title="Projection onto<br>Assistant Axis",
                orientation='h',
                x=0.5,
                xanchor='center',
                y=-0.1,
                yanchor='top',
                len=0.6,
                thickness=15
            ),
            opacity=0.7
        ),
        text=role_labels,
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>PC3: %{z:.2f}<br>Projection: %{marker.color:.2f}<extra></extra>',
        name='Role vectors'
    ))
    
    # Plot default assistant
    fig.add_trace(go.Scatter3d(
        x=[default_pc[0]],
        y=[default_pc[1]],
        z=[default_pc[2]],
        mode='markers',
        marker=dict(size=12, color='gold', symbol='diamond', line=dict(color='black', width=2)),
        text=['Default Assistant'],
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>PC3: %{z:.2f}<extra></extra>',
        name='Default Assistant'
    ))
    
    # Plot axis as extended line
    axis_direction = assistant_axis_pc[:3] / np.linalg.norm(assistant_axis_pc[:3])
    max_extent = max(np.abs(pca_result[:, :3]).max(), np.abs(default_pc[:3]).max()) * 1.5
    
    line_start = -max_extent * axis_direction
    line_end = max_extent * axis_direction
    
    fig.add_trace(go.Scatter3d(
        x=[line_start[0], line_end[0]],
        y=[line_start[1], line_end[1]],
        z=[line_start[2], line_end[2]],
        mode='lines',
        line=dict(color='red', width=8),
        name='Assistant Axis',
        hovertemplate='Assistant Axis Line<extra></extra>'
    ))
    
    # Add marker at positive end
    fig.add_trace(go.Scatter3d(
        x=[line_end[0]],
        y=[line_end[1]],
        z=[line_end[2]],
        mode='markers',
        marker=dict(size=8, color='red', symbol='diamond'),
        name='Axis (+)',
        hovertemplate='Toward Default<extra></extra>'
    ))
    
    # Add origin
    fig.add_trace(go.Scatter3d(
        x=[0],
        y=[0],
        z=[0],
        mode='markers',
        marker=dict(size=8, color='black', symbol='circle'),
        text=['Origin'],
        hovertemplate='<b>Origin (mean of roles)</b><extra></extra>',
        name='Origin'
    ))
    
    # Plot test vectors
    if test_vectors:
        colors = ['cyan', 'magenta', 'brown', 'pink', 'yellow']
        for i, (name, pc_coords) in enumerate(test_vectors.items()):
            color = colors[i % len(colors)]
            fig.add_trace(go.Scatter3d(
                x=[pc_coords[0]],
                y=[pc_coords[1]],
                z=[pc_coords[2]],
                mode='markers',
                marker=dict(size=10, color=color, symbol='x'),
                text=[name],
                hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>PC3: %{z:.2f}<extra></extra>',
                name=name
            ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title=f'PC1 ({variance_explained[0]:.1%})',
            yaxis_title=f'PC2 ({variance_explained[1]:.1%})',
            zaxis_title=f'PC3 ({variance_explained[2]:.1%})'
        ),
        width=900,
        height=750,
        template='plotly_white',
        margin=dict(b=100)  # Add bottom margin for colorbar
    )
    
    return fig

## Visualize: 2D with Extended Axis Line

In [16]:
fig_2d = plot_2d_with_axis_line(
    pca_result=pca_result,
    role_labels=role_labels,
    variance_explained=variance_explained,
    default_pc=default_pc,
    assistant_axis_pc=assistant_axis_pc,
    role_vectors_centered=role_vectors_scaled,
    assistant_axis_orig=assistant_axis[TARGET_LAYER].float().numpy(),
    title="Does Default Lie on the Axis Line? (2D)"
)

fig_2d.show()

## Visualize: 3D with Extended Axis Line

In [17]:
fig_3d = plot_3d_with_axis_line(
    pca_result=pca_result,
    role_labels=role_labels,
    variance_explained=variance_explained,
    default_pc=default_pc,
    assistant_axis_pc=assistant_axis_pc,
    role_vectors_centered=role_vectors_scaled,
    assistant_axis_orig=assistant_axis[TARGET_LAYER].float().numpy(),
    title="Does Default Lie on the Axis Line? (3D)"
)

fig_3d.show()

## Compare with Other Models

Analyze LLaMA-3.3-70B and Qwen-3-32B using their respective recommended layers.

Each model will be fitted with its own PCA space using its own role vectors.

In [18]:
# Download vectors for all models (LLaMA and Qwen)
print("Downloading vectors for all models...")

# Download LLaMA vectors (including role vectors)
print("\n1. LLaMA-3.3-70B:")
snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns=[
        "llama-3.3-70b/role_vectors/*.pt",
        "llama-3.3-70b/default_vector.pt",
        "llama-3.3-70b/assistant_axis.pt",
    ]
)
print("   ✓ Downloaded LLaMA vectors")

# Download Qwen vectors (including role vectors)
print("\n2. Qwen-3-32B:")
snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns=[
        "qwen-3-32b/role_vectors/*.pt",
        "qwen-3-32b/default_vector.pt",
        "qwen-3-32b/assistant_axis.pt",
    ]
)
print("   ✓ Downloaded Qwen vectors")

print("\n✓ All model vectors downloaded to:", local_dir)


1. LLaMA-3.3-70B:
   ✓ Downloaded LLaMA vectors

2. Qwen-3-32B:
   ✓ Downloaded Qwen vectors

✓ All model vectors downloaded to: /Users/irakl/.cache/huggingface/hub/datasets--lu-christina--assistant-axis-vectors/snapshots/3b3b788432ad33e3a28d9ff08e88a530c0740814


In [19]:
def analyze_model_full(
    model_name: str,
    target_layer: int,
    local_dir: str
) -> Tuple[np.ndarray, List[str], np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict[str, float]]:
    """
    Load role vectors, fit PCA, and analyze default vector and axis for a given model.
    
    Returns:
        pca_result: PCA-transformed role vectors
        role_labels: List of role names
        variance_explained: Variance explained by each PC
        default_pc: Default vector in PC space
        axis_pc: Axis vector in PC space
        role_vectors_centered: Mean-centered role vectors in original space
        axis_orig: Assistant axis in original space
        metrics: Dictionary of alignment metrics
    """
    print(f"\nLoading {model_name} data...")
    
    # Load role vectors
    role_vectors_dir = Path(local_dir) / model_name / "role_vectors"
    role_vectors = {}
    for pt_file in sorted(role_vectors_dir.glob("*.pt")):
        role_name = pt_file.stem
        role_vectors[role_name] = torch.load(pt_file, map_location="cpu", weights_only=False)
    
    print(f"  ✓ Loaded {len(role_vectors)} role vectors")
    
    # Extract vectors at target layer
    role_labels = list(role_vectors.keys())
    role_vectors_array = torch.stack(
        [role_vectors[label][target_layer] for label in role_labels]
    ).float().numpy()
    
    print(f"  ✓ Role vectors at layer {target_layer}: {role_vectors_array.shape}")
    
    # Load default vector and assistant axis
    default_vector_path = Path(local_dir) / model_name / "default_vector.pt"
    default_vector = torch.load(default_vector_path, map_location="cpu", weights_only=False)
    
    assistant_axis_path = Path(local_dir) / model_name / "assistant_axis.pt"
    assistant_axis = torch.load(assistant_axis_path, map_location="cpu", weights_only=False)
    
    # DIAGNOSTIC: Check relationship in ORIGINAL SPACE
    print(f"\n  DIAGNOSTIC: Vector Relationship in Original Space")
    default_orig = default_vector[target_layer].float().numpy()
    axis_orig = assistant_axis[target_layer].float().numpy()
    mean_roles_orig = role_vectors_array.mean(axis=0)
    
    computed_default_orig = mean_roles_orig + axis_orig
    diff_orig = default_orig - computed_default_orig
    relative_error = np.linalg.norm(diff_orig) / np.linalg.norm(default_orig) * 100
    
    print(f"    Relative error: {relative_error:.6f}%", end="")
    if relative_error < 1.0:
        print(" ✓")
    else:
        print(" ⚠")
    
    # Fit PCA
    scaler = MeanScaler()
    role_vectors_scaled = scaler.fit_transform(role_vectors_array)
    
    pca = PCA()
    pca_result = pca.fit_transform(role_vectors_scaled)
    variance_explained = pca.explained_variance_ratio_
    
    print(f"  ✓ PCA fitted: {variance_explained[0]:.1%}, {variance_explained[1]:.1%}, {variance_explained[2]:.1%} variance")
    
    # DIAGNOSTIC: Check PC1 alignment with Assistant Axis (in original space, not PC space)
    pc1_direction = pca.components_[0]
    # Don't center the axis - it's already a difference vector
    axis_orig_normalized = axis_orig / np.linalg.norm(axis_orig)
    pc1_normalized = pc1_direction / np.linalg.norm(pc1_direction)
    
    cosine_sim_pc1_axis = np.dot(pc1_normalized, axis_orig_normalized)
    
    print(f"  DIAGNOSTIC: PC1 vs Axis alignment: {cosine_sim_pc1_axis:.4f}", end="")
    if abs(cosine_sim_pc1_axis) > 0.71:
        print(" ✓ HIGH")
    else:
        print(" ⚠ LOW")
    
    # Project to PC space - CORRECTED: don't center the assistant axis
    default_pc = project_vector(default_vector[target_layer], scaler, pca, is_difference_vector=False)
    axis_pc = project_vector(assistant_axis[target_layer], scaler, pca, is_difference_vector=True)
    
    # Compute alignment metrics
    axis_direction = axis_pc[:3] / np.linalg.norm(axis_pc[:3])
    default_direction = default_pc[:3] / np.linalg.norm(default_pc[:3])
    
    cosine_sim = np.dot(axis_direction, default_direction)
    angle_deg = np.arccos(np.clip(cosine_sim, -1, 1)) * 180 / np.pi
    
    default_dist = np.linalg.norm(default_pc[:3])
    axis_dist = np.linalg.norm(axis_pc[:3])
    
    projection_length = np.dot(default_pc[:3], axis_direction)
    projection_point = projection_length * axis_direction
    perpendicular_dist = np.linalg.norm(default_pc[:3] - projection_point)
    
    # Check if default is at PC1 extreme
    pc1_values = pca_result[:, 0]
    max_pc1 = np.max(pc1_values)
    min_pc1 = np.min(pc1_values)
    default_pc1 = default_pc[0]
    
    at_extreme = default_pc1 > max_pc1 or default_pc1 > max_pc1 - 0.03 * (max_pc1 - min_pc1)
    
    print(f"  DIAGNOSTIC: Default at PC1 extreme: ", end="")
    if at_extreme:
        print("✓ YES")
    else:
        print("⚠ NO")
    
    # Check PC space alignment
    pc1_magnitude = abs(axis_pc[0])
    other_magnitude = np.linalg.norm(axis_pc[1:3])
    ratio = pc1_magnitude / (other_magnitude + 1e-10)
    
    print(f"  DIAGNOSTIC: Axis PC1/other ratio: {ratio:.2f}", end="")
    if ratio > 3.0:
        print(" ✓ STRONG")
    elif ratio > 1.5:
        print(" ~ MODERATE")
    else:
        print(" ⚠ WEAK")
    
    metrics = {
        'cosine_similarity': cosine_sim,
        'angle_degrees': angle_deg,
        'default_distance': default_dist,
        'axis_distance': axis_dist,
        'perpendicular_distance': perpendicular_dist,
        'projection_length': projection_length,
        'projection_percentage': projection_length / axis_dist * 100,
        'pc1_axis_cosine': cosine_sim_pc1_axis,
        'original_space_error': relative_error,
        'pc1_ratio': ratio,
    }
    
    return pca_result, role_labels, variance_explained, default_pc, axis_pc, role_vectors_scaled, axis_orig, metrics

In [20]:
print("=" * 60)
print("LLAMA-3.3-70B ANALYSIS (Layer 40)")
print("=" * 60)

llama_pca_result, llama_role_labels, llama_variance, llama_default_pc, llama_axis_pc, llama_role_vectors_centered, llama_axis_orig, llama_metrics = analyze_model_full(
    model_name="llama-3.3-70b",
    target_layer=40,  # Recommended layer for LLaMA-3.3-70B
    local_dir=local_dir
)

print(f"\nDefault vector PC coordinates (first 5): {llama_default_pc[:5]}")
print(f"Axis vector PC coordinates (first 5): {llama_axis_pc[:5]}")
print(f"\nAlignment Metrics:")
print(f"  Cosine similarity: {llama_metrics['cosine_similarity']:.4f}")
print(f"  Angle: {llama_metrics['angle_degrees']:.2f}°")
print(f"  Perpendicular distance: {llama_metrics['perpendicular_distance']:.2f}")
print(f"  Distance ratio: {llama_metrics['perpendicular_distance'] / llama_metrics['default_distance'] * 100:.1f}%")
print(f"  Projection along axis: {llama_metrics['projection_percentage']:.1f}%")

LLAMA-3.3-70B ANALYSIS (Layer 40)

Loading llama-3.3-70b data...
  ✓ Loaded 275 role vectors
  ✓ Role vectors at layer 40: (275, 8192)

  DIAGNOSTIC: Vector Relationship in Original Space
    Relative error: 0.165779% ✓
  ✓ PCA fitted: 16.6%, 15.3%, 6.6% variance
  DIAGNOSTIC: PC1 vs Axis alignment: -0.7000 ⚠ LOW
  DIAGNOSTIC: Default at PC1 extreme: ⚠ NO
  DIAGNOSTIC: Axis PC1/other ratio: 1.97 ~ MODERATE

Default vector PC coordinates (first 5): [-1.0666077   0.5254658  -0.14318752 -0.16991323 -0.2772021 ]
Axis vector PC coordinates (first 5): [-1.068109    0.52273357 -0.14317544 -0.17011534 -0.27711466]

Alignment Metrics:
  Cosine similarity: 1.0000
  Angle: 0.15°
  Perpendicular distance: 0.00
  Distance ratio: 0.3%
  Projection along axis: 100.0%


In [21]:
# Visualize LLaMA 2D
fig_llama_2d = plot_2d_with_axis_line(
    pca_result=llama_pca_result,
    role_labels=llama_role_labels,
    variance_explained=llama_variance,
    default_pc=llama_default_pc,
    assistant_axis_pc=llama_axis_pc,
    role_vectors_centered=llama_role_vectors_centered,
    assistant_axis_orig=llama_axis_orig,
    title="LLaMA-3.3-70B: Default vs Axis Alignment (2D)"
)

fig_llama_2d.show()

In [22]:
# Visualize LLaMA 3D
fig_llama_3d = plot_3d_with_axis_line(
    pca_result=llama_pca_result,
    role_labels=llama_role_labels,
    variance_explained=llama_variance,
    default_pc=llama_default_pc,
    assistant_axis_pc=llama_axis_pc,
    role_vectors_centered=llama_role_vectors_centered,
    assistant_axis_orig=llama_axis_orig,
    title="LLaMA-3.3-70B: Default vs Axis Alignment (3D)"
)

fig_llama_3d.show()

In [23]:
print("=" * 60)
print("QWEN-3-32B ANALYSIS (Layer 32)")
print("=" * 60)

qwen_pca_result, qwen_role_labels, qwen_variance, qwen_default_pc, qwen_axis_pc, qwen_role_vectors_centered, qwen_axis_orig, qwen_metrics = analyze_model_full(
    model_name="qwen-3-32b",
    target_layer=32,  # Recommended layer for Qwen-3-32B
    local_dir=local_dir
)

print(f"\nDefault vector PC coordinates (first 5): {qwen_default_pc[:5]}")
print(f"Axis vector PC coordinates (first 5): {qwen_axis_pc[:5]}")
print(f"\nAlignment Metrics:")
print(f"  Cosine similarity: {qwen_metrics['cosine_similarity']:.4f}")
print(f"  Angle: {qwen_metrics['angle_degrees']:.2f}°")
print(f"  Perpendicular distance: {qwen_metrics['perpendicular_distance']:.2f}")
print(f"  Distance ratio: {qwen_metrics['perpendicular_distance'] / qwen_metrics['default_distance'] * 100:.1f}%")
print(f"  Projection along axis: {qwen_metrics['projection_percentage']:.1f}%")

QWEN-3-32B ANALYSIS (Layer 32)

Loading qwen-3-32b data...
  ✓ Loaded 275 role vectors
  ✓ Role vectors at layer 32: (275, 5120)

  DIAGNOSTIC: Vector Relationship in Original Space
    Relative error: 0.254045% ✓
  ✓ PCA fitted: 37.9%, 13.9%, 6.9% variance
  DIAGNOSTIC: PC1 vs Axis alignment: 0.6720 ⚠ LOW
  DIAGNOSTIC: Default at PC1 extreme: ⚠ NO
  DIAGNOSTIC: Axis PC1/other ratio: 1.76 ~ MODERATE

Default vector PC coordinates (first 5): [15.395326   8.017876  -2.9134164  4.436692  -5.6163425]
Axis vector PC coordinates (first 5): [15.225055   8.1304455 -2.9197762  4.4854765 -5.688608 ]

Alignment Metrics:
  Cosine similarity: 0.9999
  Angle: 0.59°
  Perpendicular distance: 0.18
  Distance ratio: 1.0%
  Projection along axis: 100.5%


In [24]:
# Visualize Qwen 2D
fig_qwen_2d = plot_2d_with_axis_line(
    pca_result=qwen_pca_result,
    role_labels=qwen_role_labels,
    variance_explained=qwen_variance,
    default_pc=qwen_default_pc,
    assistant_axis_pc=qwen_axis_pc,
    role_vectors_centered=qwen_role_vectors_centered,
    assistant_axis_orig=qwen_axis_orig,
    title="Qwen-3-32B: Default vs Axis Alignment (2D)"
)

fig_qwen_2d.show()

In [25]:
# Visualize Qwen 3D
fig_qwen_3d = plot_3d_with_axis_line(
    pca_result=qwen_pca_result,
    role_labels=qwen_role_labels,
    variance_explained=qwen_variance,
    default_pc=qwen_default_pc,
    assistant_axis_pc=qwen_axis_pc,
    role_vectors_centered=qwen_role_vectors_centered,
    assistant_axis_orig=qwen_axis_orig,
    title="Qwen-3-32B: Default vs Axis Alignment (3D)"
)

fig_qwen_3d.show()

### Metrics Comparison

Compare alignment metrics across all three models.

### Cross-Model Comparison

In [26]:
# Create comparison table
import pandas as pd

print("="*80)
print("CROSS-MODEL COMPARISON SUMMARY")
print("="*80)

# Collect metrics for Gemma (already computed earlier)
gemma_metrics = {
    'model': 'Gemma-2-27B',
    'layer': TARGET_LAYER,
    'pc1_variance': variance_explained[0],
    'pc1_axis_cosine': cosine_sim_original,
    'original_space_error': relative_error,
    'default_at_extreme': default_pc1 > max_pc1 or default_pc1 > max_pc1 - 0.03 * (max_pc1 - min_pc1),
    'axis_pc1_ratio': abs(assistant_axis_pc[0]) / (np.linalg.norm(assistant_axis_pc[1:3]) + 1e-10),
}

comparison_data = [
    {
        'Model': 'Gemma-2-27B',
        'Layer': TARGET_LAYER,
        'PC1 Variance': f"{gemma_metrics['pc1_variance']:.1%}",
        'PC1↔Axis Cosine': f"{gemma_metrics['pc1_axis_cosine']:.3f}",
        'PC1↔Axis Aligned': '✓' if abs(gemma_metrics['pc1_axis_cosine']) > 0.71 else '⚠',
        'Formula Error': f"{gemma_metrics['original_space_error']:.3f}%",
        'Default@Extreme': '✓' if gemma_metrics['default_at_extreme'] else '⚠',
        'Axis PC1 Ratio': f"{gemma_metrics['axis_pc1_ratio']:.2f}",
    },
    {
        'Model': 'LLaMA-3.3-70B',
        'Layer': 40,
        'PC1 Variance': f"{llama_variance[0]:.1%}",
        'PC1↔Axis Cosine': f"{llama_metrics['pc1_axis_cosine']:.3f}",
        'PC1↔Axis Aligned': '✓' if abs(llama_metrics['pc1_axis_cosine']) > 0.71 else '⚠',
        'Formula Error': f"{llama_metrics['original_space_error']:.3f}%",
        'Default@Extreme': '✓' if llama_metrics.get('default_at_extreme', False) else '⚠',
        'Axis PC1 Ratio': f"{llama_metrics['pc1_ratio']:.2f}",
    },
    {
        'Model': 'Qwen-3-32B',
        'Layer': 32,
        'PC1 Variance': f"{qwen_variance[0]:.1%}",
        'PC1↔Axis Cosine': f"{qwen_metrics['pc1_axis_cosine']:.3f}",
        'PC1↔Axis Aligned': '✓' if abs(qwen_metrics['pc1_axis_cosine']) > 0.71 else '⚠',
        'Formula Error': f"{qwen_metrics['original_space_error']:.3f}%",
        'Default@Extreme': '✓' if qwen_metrics.get('default_at_extreme', False) else '⚠',
        'Axis PC1 Ratio': f"{qwen_metrics['pc1_ratio']:.2f}",
    },
]

df = pd.DataFrame(comparison_data)
print("\n" + df.to_string(index=False))

print("\n" + "="*80)
print("LEGEND:")
print("="*80)
print("PC1 Variance:      How much variance is captured by PC1")
print("PC1↔Axis Cosine:   Cosine similarity between PC1 and Assistant Axis (>0.71 = high)")
print("PC1↔Axis Aligned:  Whether PC1 aligns with the assistant axis direction")
print("Formula Error:     Error in 'default = mean(roles) + axis' (<1% = good)")
print("Default@Extreme:   Whether default assistant is at PC1 extreme")
print("Axis PC1 Ratio:    Ratio of axis PC1 component to other components (>3.0 = strong)")

print("\n" + "="*80)
print("KEY FINDINGS:")
print("="*80)

# Analyze findings
models_with_high_alignment = sum(1 for d in comparison_data if d['PC1↔Axis Aligned'] == '✓')
print(f"\n1. PC1-Axis Alignment:")
print(f"   {models_with_high_alignment}/3 models show high PC1↔Axis alignment (>0.71)")

if models_with_high_alignment == 0:
    print("   ⚠ None of the models at their respective layers show strong PC1-axis alignment")
    print("   This suggests that for these specific layers, the assistant axis is NOT")
    print("   the dominant source of variance in the role vectors.")
elif models_with_high_alignment == 3:
    print("   ✓ All models show strong PC1-axis alignment")
    print("   The assistant axis is the dominant source of variance.")
else:
    print("   ~ Mixed results across models")

print(f"\n2. Vector Formula Validation:")
all_formulas_valid = all(float(d['Formula Error'].rstrip('%')) < 1.0 for d in comparison_data)
if all_formulas_valid:
    print("   ✓ All models satisfy: default ≈ mean(roles) + axis")
    print("   The pre-computed vectors are mathematically consistent.")
else:
    print("   ⚠ Some models show formula mismatch")

print("\nNote: The three models have different architectures and cannot be")
print("directly compared in the same PCA space. Each model was analyzed")
print("independently using its own role vectors and PCA decomposition.")

CROSS-MODEL COMPARISON SUMMARY

        Model  Layer PC1 Variance PC1↔Axis Cosine PC1↔Axis Aligned Formula Error Default@Extreme Axis PC1 Ratio
  Gemma-2-27B     22        48.8%          -0.660                ⚠        0.129%               ⚠           9.36
LLaMA-3.3-70B     40        16.6%          -0.700                ⚠        0.166%               ⚠           1.97
   Qwen-3-32B     32        37.9%           0.672                ⚠        0.254%               ⚠           1.76

LEGEND:
PC1 Variance:      How much variance is captured by PC1
PC1↔Axis Cosine:   Cosine similarity between PC1 and Assistant Axis (>0.71 = high)
PC1↔Axis Aligned:  Whether PC1 aligns with the assistant axis direction
Formula Error:     Error in 'default = mean(roles) + axis' (<1% = good)
Default@Extreme:   Whether default assistant is at PC1 extreme
Axis PC1 Ratio:    Ratio of axis PC1 component to other components (>3.0 = strong)

KEY FINDINGS:

1. PC1-Axis Alignment:
   0/3 models show high PC1↔Axis alignment 

## Summary

This notebook demonstrates:

✅ **Loading pre-computed vectors** from HuggingFace  
✅ **Using assistant_axis utilities** (`load_axis`, `ProbingModel`)  
✅ **PCA landscape visualization** with interactive plots  
✅ **Alignment analysis** - checking if default lies on the axis line  
✅ **Extended axis visualization** - showing the axis as a bidirectional line  
✅ **Multi-model analysis** - analyzing Gemma-2-27B, LLaMA-3.3-70B, and Qwen-3-32B independently  

**Key insights:**
1. The assistant axis represents the direction from role-playing behavior toward default assistant behavior
2. By extending it as a line through the origin, you can see whether the default assistant vector aligns with this direction
3. The perpendicular distance tells you how "off-axis" a persona is
4. Each model was analyzed using its recommended layer:
   - Gemma-2-27B: Layer 22 (46 layers × 4608 dims)
   - LLaMA-3.3-70B: Layer 40 (80 layers × 8192 dims)
   - Qwen-3-32B: Layer 32 (64 layers × 5120 dims)
5. Models have different architectures and are analyzed in separate PCA spaces